## Import Library

In [ ]:
"""CLIP prototype calibration script for NIH ChestX-ray14 distribution.

Extracts CLIP embeddings from 500 stratified NIH samples, computes the
prototype centroid and Euclidean distance threshold for input validation.

Run once on Kaggle. Output: clip_prototype.json
Save to models/weights/ alongside multilabel_model.pt.

References:
  - Radford et al. (2021): CLIP, ICML 2021.
  - Snell et al. (2017): Prototypical Networks, NeurIPS 2017.
    (Euclidean distance preferred over cosine for prototype classification.)
"""

# pip install git+https://github.com/openai/CLIP.git

import json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import clip
from PIL import Image

## Config

In [ ]:
DATA_ROOT   = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")
CSV_PATH    = DATA_ROOT / "Data_Entry_2017.csv"
IMAGE_DIRS  = [DATA_ROOT / f"images_{i:03d}/images" for i in range(1, 13)]
OUTPUT_PATH = Path("/kaggle/working/clip_prototype.json")

N_NO_FINDING   = 250   # "No Finding" samples
N_PER_CONDTION = 18   # ~250 total across 14 conditions (18 × 14 ≈ 252)
K_THRESHOLD    = 1.5   # τ = mean_dist - k × std_dist (Snell et al., 2017)
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

## Helper Function

In [ ]:
# Build image index 
def build_image_index() -> dict:
    """Precompute filename→path mapping across all 12 image subdirectories."""
    index = {}
    for d in IMAGE_DIRS:
        if d.exists():
            for p in d.iterdir():
                index[p.name] = p
    return index


# Stratified sampling 
def sample_stratified(image_index: dict) -> list[Path]:
    """Sample 500 images: 250 No Finding + ~250 across 14 conditions."""
    df = pd.read_csv(CSV_PATH, usecols=["Image Index", "Finding Labels"])
    df = df[df["Image Index"].isin(image_index)]

    all_conditions = [
        "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
        "Effusion", "Emphysema", "Fibrosis", "Hernia",
        "Infiltration", "Mass", "Nodule", "Pleural_Thickening",
        "Pneumonia", "Pneumothorax",
    ]

    sampled_paths: list[Path] = []

    # Stratum 1: No Finding
    no_finding = df[df["Finding Labels"] == "No Finding"].sample(
        n=N_NO_FINDING, random_state=42
    )
    sampled_paths.extend(image_index[fn] for fn in no_finding["Image Index"])

    # Stratum 2: each condition
    for cond in all_conditions:
        positive = df[df["Finding Labels"].str.contains(cond, na=False)]
        n = min(N_PER_CONDTION, len(positive))
        if n > 0:
            subset = positive.sample(n=n, random_state=42)
            sampled_paths.extend(image_index[fn] for fn in subset["Image Index"])

    print(f"Total samples collected: {len(sampled_paths)}")
    return sampled_paths


# CLIP embedding extraction 
def extract_embeddings(
    paths: list[Path],
    clip_model,
    preprocess,
) -> np.ndarray:
    """Extract L2-normalized CLIP image embeddings for all sample paths."""
    embeddings = []
    clip_model.eval()

    with torch.no_grad():
        for path in paths:
            try:
                with Image.open(path) as img:
                    tensor = preprocess(img.convert("RGB")).unsqueeze(0).to(DEVICE)
                emb = clip_model.encode_image(tensor)
                emb = emb / emb.norm(dim=-1, keepdim=True)   # L2 normalize
                embeddings.append(emb.cpu().numpy().flatten())
            except Exception as e:
                print(f"Skipped {path.name}: {e}")

    return np.array(embeddings, dtype=np.float32)  # (N, 512)


# Calibration 
def calibrate(embeddings: np.ndarray) -> dict:
    """Compute centroid and adaptive Euclidean distance threshold.

    Threshold formula (Snell et al., 2017):
        τ = mean_dist + k × std_dist
    Images with dist(q, centroid) > τ are rejected.
    k=1.5 is conservative — flags outliers beyond 1.5σ from mean.
    """
    centroid = embeddings.mean(axis=0)                          # (512,)
    dists = np.linalg.norm(embeddings - centroid, axis=1)   # (N,)

    mean_dist = float(dists.mean())
    std_dist = float(dists.std())
    threshold = mean_dist + K_THRESHOLD * std_dist             # upper bound

    print(f"Centroid norm     : {np.linalg.norm(centroid):.4f}")
    print(f"Mean distance     : {mean_dist:.4f}")
    print(f"Std distance      : {std_dist:.4f}")
    print(f"Threshold (k=1.5) : {threshold:.4f}")

    return {
        "centroid":   centroid.tolist(),
        "threshold":  round(threshold, 6),
        "mean_dist":  round(mean_dist, 6),
        "std_dist":   round(std_dist, 6),
        "n_samples":  len(embeddings),
        "k":          K_THRESHOLD,
    }

## Main Run

In [ ]:
#  Entry point
if __name__ == "__main__":
    print(f"Device: {DEVICE}")

    clip_model, preprocess = clip.load("ViT-B/32", device=DEVICE)
    print("CLIP model loaded")

    image_index = build_image_index()
    print(f"Image index: {len(image_index)} files")

    sample_paths = sample_stratified(image_index)
    embeddings = extract_embeddings(sample_paths, clip_model, preprocess)
    print(f"Embeddings shape: {embeddings.shape}")

    prototype = calibrate(embeddings)

    with open(OUTPUT_PATH, "w") as f:
        json.dump(prototype, f, indent=2)

    print(f"Prototype saved: {OUTPUT_PATH}")